# 05 — Watchdog: Keep Fabric Mirroring Healthy

Invoked by a Data Pipeline on a schedule (recommended: every 5–10 min).
Checks the Mirrored Database status via REST API and restarts it if stopped or in error.

| Status | Action |
|---|---|
| `Running` | None |
| `Stopped` | `startMirroring` |
| `Failed` | `startMirroring` |
| `Initializing` | Wait — do not intervene |
| `Replicating` | None (transient OK) |

> **Note:** `startMirroring` in `Failed` state may trigger a full re-initialization if CDC is down on SQL Server.  
> Ensure the SQL Agent is running on the VM before relying on this watchdog.

**Requires `silver_rti` as the default lakehouse** (to write `WatchdogLog`).

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
MIRRORED_DB_NAME = "GasPlantDB"   # display name of the Mirrored Database item in Fabric workspace
SILVER_LAKEHOUSE = "silver_rti"   # default lakehouse for WatchdogLog

In [ ]:
# ── Auth and workspace ────────────────────────────────────────────────────────
# spark.conf 'trident.workspace.id' is injected by Fabric into every Spark session
import requests
import notebookutils
from datetime import datetime, timezone

workspace_id = spark.conf.get("trident.workspace.id")
token        = notebookutils.credentials.getToken("https://api.fabric.microsoft.com/")
headers      = {
    "Authorization": f"Bearer {token}",
    "Content-Type":  "application/json",
}
base_url = "https://api.fabric.microsoft.com/v1"

now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
print(f"[{now}] Workspace ID : {workspace_id}")

In [ ]:
# ── Find Mirrored Database item ID ────────────────────────────────────────────
resp = requests.get(f"{base_url}/workspaces/{workspace_id}/mirroredDatabases", headers=headers)
resp.raise_for_status()

mirrored_dbs = resp.json().get("value", [])
db_item = next(
    (db for db in mirrored_dbs if db["displayName"] == MIRRORED_DB_NAME),
    None,
)

if db_item is None:
    raise RuntimeError(f"Mirrored Database '{MIRRORED_DB_NAME}' not found in workspace {workspace_id}.")

db_id = db_item["id"]
print(f"Mirrored DB  : {MIRRORED_DB_NAME} ({db_id})")

In [ ]:
# ── Get mirroring status ──────────────────────────────────────────────────────
status_url = f"{base_url}/workspaces/{workspace_id}/mirroredDatabases/{db_id}/getMirroringStatus"
resp = requests.post(status_url, headers=headers, json={})   # Fabric uses POST for action endpoints
resp.raise_for_status()

status_payload = resp.json()
status         = status_payload.get("status", "Unknown")

# Table-level detail (if present)
tables = status_payload.get("tables", [])
failed_tables = [t for t in tables if t.get("status") not in ("Replicating", "Initialized", "Initializing")]

print(f"Mirroring status : {status}")
if tables:
    print(f"Tables total     : {len(tables)}")
    print(f"Tables with issue: {len(failed_tables)}")
    for t in failed_tables:
        print(f"  - {t.get('tableName', '?')} → {t.get('status', '?')}")

In [ ]:
# ── Restart mirroring if stopped or failed ────────────────────────────────────
HEALTHY_STATUSES  = {"Running", "Initializing", "Replicating"}
RESTART_STATUSES  = {"Stopped", "Failed", "Inactive", "Unknown"}

action_taken  = "none"
action_detail = f"status was '{status}'"

if status in HEALTHY_STATUSES:
    print(f"Mirroring is healthy ('{status}'). Nothing to do.")

elif status in RESTART_STATUSES:
    print(f"Mirroring is '{status}'. Attempting startMirroring...")
    start_url = f"{base_url}/workspaces/{workspace_id}/mirroredDatabases/{db_id}/startMirroring"
    resp = requests.post(start_url, headers=headers, json={})

    if resp.status_code in (200, 202):
        action_taken  = "startMirroring"
        action_detail = f"triggered from status '{status}' - HTTP {resp.status_code}"
        print(f"startMirroring accepted. HTTP {resp.status_code}")
        print("Mirroring will re-initialize; allow 2-5 min for status to become 'Running'.")
    else:
        action_taken  = "startMirroring_failed"
        action_detail = f"HTTP {resp.status_code}: {resp.text}"
        raise RuntimeError(f"Failed to start mirroring. {action_detail}")

else:
    action_detail = f"unrecognized status '{status}'"
    print(f"Unrecognized status '{status}'. No action taken - review manually.")

In [ ]:
# ── Write watchdog run result to WatchdogLog (silver_rti) ─────────────────────
from pyspark.sql import Row
from datetime import datetime, timezone

log_row = Row(
    run_ts       = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
    watchdog     = "mirroring",
    status_found = status,
    action_taken = action_taken,
    details      = action_detail,
)

spark.createDataFrame([log_row]) \
     .write.format("delta") \
     .mode("append") \
     .saveAsTable("WatchdogLog")

print(f"WatchdogLog written: watchdog=mirroring | status_found={status} | action_taken={action_taken}")